In [1]:
import pandas as pd

df = pd.read_csv("quality_data.csv")

# Create input text
df["input_text"] = (
    "narration: " + df["narration"].astype(str) +
    " | mode: " + df["mode"].astype(str) +
    " | type: " + df["type"].astype(str)
)

df["input_text"] = df["input_text"].str.lower()

# Create label
df["label"] = df["category"] + "_" + df["subcategory"]

df = df[["input_text", "label", "narration", "mode", "type", "category", "subcategory"]]

df.head()

,input_text,label,narration,mode,type,category,subcategory
0,narration: interest from fixed deposit | mode:...,investment_fd_interest,interest from fixed deposit,NEFT,Credit,investment,fd_interest
1,narration: fd interest payout | mode: neft | t...,investment_fd_interest,FD interest payout,NEFT,Credit,investment,fd_interest
2,narration: fixed deposit interest | mode: neft...,investment_fd_interest,Fixed deposit interest,NEFT,Credit,investment,fd_interest
3,narration: bank interest received | mode: neft...,investment_fd_interest,bank interest received,NEFT,Credit,investment,fd_interest
4,narration: fd maturity interest | mode: neft |...,investment_fd_interest,FD maturity interest,NEFT,Credit,investment,fd_interest


In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    df["input_text"],
    df["label"],
    df,   # 👈 IMPORTANT
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [3]:
from gliner import GLiNER

# GLiNER2 model
gliner_model = GLiNER.from_pretrained("urchade/gliner_large-v2")

labels = list(set(y_train))

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\utils\_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [4]:
def gliner_predict(text):
    entities = gliner_model.predict_entities(text, labels)

    if entities:
        best = max(entities, key=lambda x: x["score"])
        return best["label"], best["score"]

    return "unknown_unknown", 0.0

In [7]:
y_pred = []
conf_scores = []

for text in X_test:
    label, score = gliner_predict(text)
    y_pred.append(label)
    conf_scores.append(score)

In [8]:
result_df = df_test.copy()

result_df["predicted_label"] = y_pred
result_df["confidence"] = conf_scores

In [9]:
result_df["confidence_percent"] = (result_df["confidence"] * 100).round(2).astype(str) + "%"

def confidence_level(c):
    if c >= 0.6:
        return "High"
    elif c >= 0.3:
        return "Medium"
    else:
        return "Low"

result_df["confidence_level"] = result_df["confidence"].apply(confidence_level)

In [12]:
# Actual split
result_df[["category", "subcategory"]] = result_df["label"].str.split("_", n=1, expand=True)

# Predicted split
result_df[["predicted_category", "predicted_subcategory"]] = result_df["predicted_label"].str.split("_", n=1, expand=True)

In [14]:
result_df["correct"] = result_df["label"] == result_df["predicted_label"]

In [11]:
print(result_df.columns)

Index(['input_text', 'label', 'narration', 'mode', 'type', 'category',
       'subcategory', 'predicted_label', 'confidence', 'confidence_percent',
       'confidence_level'],
      dtype='object')


In [15]:
final_df = result_df[
    [
        "narration",
        "mode",
        "type",
        "category",
        "subcategory",
        "predicted_category",
        "predicted_subcategory",
        "confidence",
        "confidence_percent",
        "confidence_level",
        "correct"
    ]
]

final_df.head(20)

,narration,mode,type,category,subcategory,predicted_category,predicted_subcategory,confidence,confidence_percent,confidence_level,correct
84,Home loan EMI payment,AUTO_DEBIT,Debit,expense,loan,expense,loan,0.975774,97.58%,High,True
49,Shopping payment online,UPI,Debit,expense,shopping,expense,shopping,0.893196,89.32%,High,True
56,Broadband bill payment,UPI,Debit,expense,bills,expense,bills,0.772989,77.3%,High,True
42,Amazon purchase,UPI,Debit,expense,shopping,expense,shopping,0.966225,96.62%,High,True
78,Policy premium debit,AUTO_DEBIT,Debit,expense,insurance,expense,insurance,0.874656,87.47%,High,True
66,Fastag recharge,UPI,Debit,expense,transport,expense,loan,0.747515,74.75%,High,False
52,Internet bill payment,UPI,Debit,expense,bills,expense,bills,0.804428,80.44%,High,True
26,Salary credited,NEFT,Credit,income,salary,unknown,unknown,0.000000,0.0%,Low,False
79,Insurance renewal payment,AUTO_DEBIT,Debit,expense,insurance,expense,insurance,0.836774,83.68%,High,True
41,Online food order,UPI,Debit,expense,food,expense,shopping,0.827575,82.76%,High,False


In [16]:
from sklearn.metrics import classification_report
import numpy as np

report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()

# F2 Score
def f2_score(p, r):
    if (p + r) == 0:
        return 0
    return (5 * p * r) / (4 * p + r)

report_df["f2_score"] = report_df.apply(
    lambda row: f2_score(row["precision"], row["recall"]),
    axis=1
)

report_df = report_df.round(3)

print("\nClassification Report:")
print(report_df)


Classification Report:
                        precision  recall  f1-score  support  f2_score
expense_bills               1.000   1.000     1.000    2.000     1.000
expense_food                0.000   0.000     0.000    1.000     0.000
expense_health              0.000   0.000     0.000    1.000     0.000
expense_insurance           1.000   1.000     1.000    2.000     1.000
expense_loan                0.333   1.000     0.500    2.000     0.714
expense_shopping            0.667   1.000     0.800    2.000     0.909
expense_transport           1.000   0.500     0.667    2.000     0.556
income_salary               0.000   0.000     0.000    2.000     0.000
investment_fd_booking       1.000   1.000     1.000    2.000     1.000
investment_fd_interest      1.000   0.500     0.667    2.000     0.556
investment_stocks           1.000   1.000     1.000    1.000     1.000
unknown_unknown             0.000   0.000     0.000    0.000     0.000
accuracy                    0.684   0.684     0.684  

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modif

In [17]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("\nOverall Metrics:")
print("Accuracy:", round(accuracy, 3))
print("Macro F1:", report_df.loc["macro avg", "f1-score"])
print("Weighted F1:", report_df.loc["weighted avg", "f1-score"])


Overall Metrics:
Accuracy: 0.684
Macro F1: 0.553
Weighted F1: 0.646


In [18]:
filtered_df = report_df.drop(["accuracy", "macro avg", "weighted avg"], errors="ignore")

best = filtered_df["f1-score"].idxmax()
worst = filtered_df["f1-score"].idxmin()

print("\nBest Performing Category:")
print(best, filtered_df.loc[best, "f1-score"])

print("\nWorst Performing Category:")
print(worst, filtered_df.loc[worst, "f1-score"])


Best Performing Category:
expense_bills 1.0

Worst Performing Category:
expense_food 0.0
